## 📝 Exercises

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("RDD Lab") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark Session and Context successfully created")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 20:05:03 WARN Utils: Your hostname, codespaces-ff5ee5, resolves to a loopback address: 127.0.0.1; using 10.0.15.142 instead (on interface eth0)
26/06/21 20:05:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 20:05:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session and Context successfully created


### Task 1
Parse text data into structured format

In [4]:
# 1. Read the file using standard Python
with open("employees.txt", "r") as f:
    local_lines = f.readlines()

In [5]:
# 2. Parallelize the plain Python list into a Spark RDD
raw_employees_rdd = sc.parallelize(local_lines)

In [6]:
raw_employees_rdd.collect()

['emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp\n',
 '1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8\n',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5\n',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3\n',
 '4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7\n',
 '\n',
 '5,David Jones,Finance,Senior Analyst,105000,Boston,2021-08-12,4.3,6\n',
 '6,Lisa Garcia,IT,DevOps Engineer,115000,Seattle,2022-04-18,4.6,4\n',
 '7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5\n',
 '8,Patricia Wilson,HR,HR Manager,88000,Denver,2020-02-28,4.1,6\n',
 '\n',
 '9,James Anderson,Sales,Sales Manager,110000,Atlanta,2021-12-01,4.4,7\n',
 '10,Mary Thomas,Engineering,Tech Lead,145000,Seattle,2018-07-14,4.9,12']

In [7]:
print(raw_employees_rdd.id())

0


In [8]:
# 3. Extract the header text string
header = raw_employees_rdd.first()

In [9]:
print(header)

emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp



In [10]:
# 4. Clean the data: filter out blank lines and remove the header line
emp_rdd = raw_employees_rdd.filter(lambda line: len(line.strip()) > 0 and line.strip() != header.strip())

In [11]:
emp_rdd.collect()

['1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8\n',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5\n',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3\n',
 '4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7\n',
 '5,David Jones,Finance,Senior Analyst,105000,Boston,2021-08-12,4.3,6\n',
 '6,Lisa Garcia,IT,DevOps Engineer,115000,Seattle,2022-04-18,4.6,4\n',
 '7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5\n',
 '8,Patricia Wilson,HR,HR Manager,88000,Denver,2020-02-28,4.1,6\n',
 '9,James Anderson,Sales,Sales Manager,110000,Atlanta,2021-12-01,4.4,7\n',
 '10,Mary Thomas,Engineering,Tech Lead,145000,Seattle,2018-07-14,4.9,12']

In [12]:
# 5. Map each line into a list by splitting on commas
structured_emp_rdd = emp_rdd.map(lambda line: line.strip().split(','))

In [13]:
print(structured_emp_rdd.id())

3


In [14]:
# Take 3 sample records to inspect the list structure
for record in structured_emp_rdd.take(3):
    print(record)

['1', 'John Smith', 'Engineering', 'Senior Developer', '125000', 'San Francisco', '2021-03-15', '4.5', '8']
['2', 'Sarah Johnson', 'Sales', 'Account Executive', '85000', 'New York', '2022-01-10', '4.2', '5']
['3', 'Michael Williams', 'Engineering', 'Software Engineer', '95000', 'Austin', '2023-06-20', '3.8', '3']


In [15]:
structured_emp_rdd_1 = emp_rdd.flatMap(lambda x: x.split(" "))

In [16]:
print(structured_emp_rdd_1.id())

5


In [17]:
structured_emp_rdd_1.collect()

['1,John',
 'Smith,Engineering,Senior',
 'Developer,125000,San',
 'Francisco,2021-03-15,4.5,8\n',
 '2,Sarah',
 'Johnson,Sales,Account',
 'Executive,85000,New',
 'York,2022-01-10,4.2,5\n',
 '3,Michael',
 'Williams,Engineering,Software',
 'Engineer,95000,Austin,2023-06-20,3.8,3\n',
 '4,Jennifer',
 'Brown,Marketing,Marketing',
 'Manager,92000,Chicago,2020-11-05,4.7,7\n',
 '5,David',
 'Jones,Finance,Senior',
 'Analyst,105000,Boston,2021-08-12,4.3,6\n',
 '6,Lisa',
 'Garcia,IT,DevOps',
 'Engineer,115000,Seattle,2022-04-18,4.6,4\n',
 '7,Robert',
 'Martinez,Legal,Legal',
 'Counsel145000,San',
 'Francisco,2019-09-22,4.8,10,',
 '5\n',
 '8,Patricia',
 'Wilson,HR,HR',
 'Manager,88000,Denver,2020-02-28,4.1,6\n',
 '9,James',
 'Anderson,Sales,Sales',
 'Manager,110000,Atlanta,2021-12-01,4.4,7\n',
 '10,Mary',
 'Thomas,Engineering,Tech',
 'Lead,145000,Seattle,2018-07-14,4.9,12']

### Task 2
Count occurrences of each Department



In [18]:
# Map to pairs and reduce by key to aggregate counts
dept_counts_rdd = structured_emp_rdd.map(lambda x: (x[2], 1)).reduceByKey(lambda a, b: a + b)

In [19]:
for dept, count in dept_counts_rdd.collect():
    print(f"Department: {dept:<15} Count: {count}")

Department: Engineering     Count: 3
Department: Sales           Count: 2
Department: Finance         Count: 1
Department: IT              Count: 1
Department: Legal           Count: 1
Department: HR              Count: 1
Department: Marketing       Count: 1


### Task 3
Filter invalid records safely


In [20]:
def is_valid_record(row):
    # 1. First check structural length
    if len(row) != 9:
        return False
    
    # 2. Try to parse numbers dynamically to detect corrupted strings
    try:
        float(row[4])  # Salary check 
        float(row[7])  # Performance rating check
        float(row[8])  # Years of experience check
        return True
    
    except ValueError:
        return False

In [21]:
# Filter using our smart dynamic validator
valid_emp_rdd = structured_emp_rdd.filter(is_valid_record)
invalid_emp_rdd = structured_emp_rdd.filter(lambda x: not is_valid_record(x))

In [22]:
print(f"Total clean records kept: {valid_emp_rdd.count()}")
print("\nCorrupted rows successfully caught and removed:")
print(f"Total bad records removed: {invalid_emp_rdd.count()}")
for bad_row in invalid_emp_rdd.collect():
    print(bad_row)

Total clean records kept: 9

Corrupted rows successfully caught and removed:
Total bad records removed: 1
['7', 'Robert Martinez', 'Legal', 'Legal Counsel145000', 'San Francisco', '2019-09-22', '4.8', '10', ' 5']


### Task 4
Calculate the average salary per department

In [23]:
# 1. Map to key-value pairs: (Department, (Salary, 1))
# index 2 is Department and index 4 is Salary (converted to float)
dept_salary_pairs = valid_emp_rdd.map(lambda x: (x[2], (float(x[4]), 1)))

In [24]:
dept_salary_pairs.collect()

[('Engineering', (125000.0, 1)),
 ('Sales', (85000.0, 1)),
 ('Engineering', (95000.0, 1)),
 ('Marketing', (92000.0, 1)),
 ('Finance', (105000.0, 1)),
 ('IT', (115000.0, 1)),
 ('HR', (88000.0, 1)),
 ('Sales', (110000.0, 1)),
 ('Engineering', (145000.0, 1))]

In [25]:
# 2. Reduce by key to sum up the total salaries and total employee counts per dept
# (total_salary_a + total_salary_b, count_a + count_b)
dept_totals = dept_salary_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

In [26]:
dept_totals.collect()

[('Engineering', (365000.0, 3)),
 ('Sales', (195000.0, 2)),
 ('Finance', (105000.0, 1)),
 ('IT', (115000.0, 1)),
 ('HR', (88000.0, 1)),
 ('Marketing', (92000.0, 1))]

In [27]:
# 3. Map values to calculate the final average: total_salary / count
dept_averages = dept_totals.mapValues(lambda val: round(val[0] / val[1], 2))

In [28]:
dept_averages.collect()

[('Engineering', 121666.67),
 ('Sales', 97500.0),
 ('Finance', 105000.0),
 ('IT', 115000.0),
 ('HR', 88000.0),
 ('Marketing', 92000.0)]

In [29]:
for dept, avg_sal in dept_averages.collect():
    print(f"Department: {dept:<15} Average Salary: ${avg_sal:,.2f}")

Department: Engineering     Average Salary: $121,666.67
Department: Sales           Average Salary: $97,500.00
Department: Finance         Average Salary: $105,000.00
Department: IT              Average Salary: $115,000.00
Department: HR              Average Salary: $88,000.00
Department: Marketing       Average Salary: $92,000.00
